# Adult-cencus-income prediction using Stacking and Blending

In [1]:
import pandas as pd
# column names
columns = [
    'age', 'workclass', 'fnlwgt', 'education',
    'education-num', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'capital-gain',
    'capital-loss', 'hours-per-week', 'native-country',
    'income'
]

# load dataset
df = pd.read_csv(
    '../data/adult.data',
    header=None,
    names=columns,
    skipinitialspace=True
)
print("Dataset shape:", df.shape)

Dataset shape: (32561, 15)


In [2]:
X = df.drop(columns='income')
y = df['income']

# encode target
y = (y == '>50K').astype(int)

In [8]:
from sklearn.model_selection import train_test_split

# split train/valid/test
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

## Stacking

In [11]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

estimators = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=200, random_state=42)),
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=5
)

# Convert categorical variables to numeric dummy columns
X_train_encoded = pd.get_dummies(X_train)
X_valid_encoded = pd.get_dummies(X_valid)

# Align columns to ensure X_train and X_valid have the exact same columns
X_train_encoded, X_valid_encoded = X_train_encoded.align(
    X_valid_encoded, join="left", axis=1, fill_value=0
)

# Now fit your stacking classifier
stack.fit(X_train_encoded, y_train)
y_pred = stack.predict(X_valid_encoded)

In [14]:
from sklearn.metrics import accuracy_score

print(f'Accuracy score: {accuracy_score(y_valid, y_pred)}') # 0.8669865642994242

Accuracy score: 0.8669865642994242


## Blending

In [17]:
# train models
model1 = RandomForestClassifier(n_estimators=200, random_state=42)
model2 = XGBClassifier(n_estimators=200, random_state=42)
model3 = LogisticRegression(random_state=42, max_iter=1000)

# fit models
model1.fit(X_train_encoded, y_train)
model2.fit(X_train_encoded, y_train)
model3.fit(X_train_encoded, y_train)

# 3. predict probabilities on the validation data
p1 = model1.predict_proba(X_valid_encoded)[:, 1]
p2 = model2.predict_proba(X_valid_encoded)[:, 1]
p3 = model3.predict_proba(X_valid_encoded)[:, 1]

# 4. Compute the weighted blend (weights must sum to 1.0: 0.5 + 0.3 + 0.2 = 1.0)
blend = (0.5 * p1) + (0.3 * p2) + (0.2 * p3)

print(f"Blended Probabilities: {blend}")

Blended Probabilities: [0.84944365 0.1040133  0.19561408 ... 0.57194265 0.99934598 0.18454932]


/Users/hofman/Documents/najot_talim/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [18]:
# Convert probabilities to 0 or 1 based on a 0.5 threshold
y_pred_blend = (blend >= 0.5).astype(int)

# evaluate
print("Blend Accuracy:", accuracy_score(y_valid, y_pred_blend))

Blend Accuracy: 0.8666026871401151


# Analiz

In [19]:
# Evaluation function
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

In [20]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report)

# Stacking probabilities
stack_prob = stack.predict_proba(X_valid_encoded)[:, 1]

# Blending predictions (default threshold = 0.5)
blend_pred = (blend >= 0.5).astype(int)
stack_pred = (stack_prob >= 0.5).astype(int)



# Compare models
results = []

results.append(evaluate_model(
    'Stacking', y_valid, stack_pred, stack_prob
))

results.append(evaluate_model(
    'Blending', y_valid, blend_pred, blend
))

results_df = pd.DataFrame(results)

print(results_df.round(4))
#       Model  Accuracy  Precision  Recall  F1-score  ROC-AUC
# 0  Stacking    0.8670     0.7651  0.6462    0.7006   0.9174
# 1  Blending    0.8666     0.7677  0.6398    0.6980   0.9134
# barcha metrika boyicha stacking yutib chiqdi

      Model  Accuracy  Precision  Recall  F1-score  ROC-AUC
0  Stacking    0.8670     0.7651  0.6462    0.7006   0.9174
1  Blending    0.8666     0.7677  0.6398    0.6980   0.9134
